# Predict the next AFL action and location

Given the actions observed so far in a team's possession, predict the next retained action and where it will occur. This notebook explains the dataset and reviews a Seq2Event-style Transformer alongside two simple baselines.

The action head includes `end_possession`. Its location head is evaluated only when the possession continues. The model is shared across teams and is not yet conditioned on a particular opponent. This is a separate task from predicting whether a possession will eventually produce a shot or goal.

This notebook reads the committed example report and does not retrain. See [the specification](../docs/next-event-prediction.md) for training commands, data rules and differences from the original paper.

## 1. Locate the saved report

Find the repository from the current directory and read the exported results. The output identifies the report using a repository-relative path. A fresh clone can run these report cells without downloading the raw data or model weights.

In [1]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)

project_root = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file()),
    None,
)
if project_root is None:
    raise RuntimeError("Open the notebook from the repository root or notebooks directory.")
report_dir = project_root / "reports" / "examples" / "next-event-v1"
summary = json.loads((report_dir / "summary.json").read_text(encoding="utf-8"))
audit = json.loads((report_dir / "dataset_audit.json").read_text(encoding="utf-8"))
print("Report:", report_dir.relative_to(project_root).as_posix())
print("Task: next action and location")
print("Test scope: already-observed 2026 benchmark")

Report: reports/examples/next-event-v1
Task: next action and location
Test scope: already-observed 2026 benchmark


## 2. Inspect an actual history-to-target sequence

Each row below is an observation and its target. The action target is taken from the **next retained event**, not the next raw annotation. A kick-result annotation, for example, does not become a second action.

All rows in this example belong to one possession. The final `end_possession` target has no location. Its stored masked placeholder is displayed as missing rather than as the centre of the ground.

In [2]:
example = pd.read_csv(report_dir / "example_possession.csv")
shown = example[["event_index", "action_type", "x_norm", "y_norm",
                 "target_action", "target_x", "target_y", "has_location"]].copy()
shown.loc[~shown.has_location, ["target_x", "target_y"]] = np.nan
display(shown.round(3))

,event_index,action_type,x_norm,y_norm,target_action,target_x,target_y,has_location
0,0,free_kick,-0.012,0.057,handball,0.000,0.028,True
1,1,handball,0.000,0.028,receive,0.050,-0.071,True
2,2,receive,0.050,-0.071,kick,0.138,-0.085,True
3,3,kick,0.138,-0.085,loose_ball_get,0.675,-0.128,True
4,4,loose_ball_get,0.675,-0.128,handball,0.675,-0.142,True
5,5,handball,0.675,-0.142,receive,0.700,-0.213,True
6,6,receive,0.700,-0.213,kick,0.750,-0.255,True
7,7,kick,0.750,-0.255,end_possession,NaN,NaN,False


## 3. See the growing input windows

At the first action, the model receives one event. At the second, it receives two, and so on, up to the selected history cap. It then uses the most recent events. The target event never enters its own input window.

Histories reset at possession boundaries, keeping all coordinates in the same team's attacking frame. An opponent gaining control produces `end_possession`, rather than a target location expressed in the opponent's reversed coordinate frame.

In [3]:
chosen_index = summary["selected"]["candidate"]
config = summary["plan"]["configs"][chosen_index]
cap = config["max_history"]
windows = []
for end, row in example.iterrows():
    history = example.iloc[max(0, end - cap + 1):end + 1]
    windows.append({
        "after_event": int(row.event_index) + 1,
        "observed_history": " -> ".join(history.action_type),
        "target_action": row.target_action,
        "location_is_scored": bool(row.has_location),
    })
print("Selected maximum history:", cap)
display(pd.DataFrame(windows))

Selected maximum history: 32


,after_event,observed_history,target_action,location_is_scored
0,1,free_kick,handball,True
1,2,free_kick -> handball,receive,True
2,3,free_kick -> handball -> receive,kick,True
3,4,free_kick -> handball -> receive -> kick,loose_ball_get,True
4,5,free_kick -> handball -> receive -> kick -> loose_ball_get,handball,True
5,6,free_kick -> handball -> receive -> kick -> loose_ball_get -> handball,receive,True
6,7,free_kick -> handball -> receive -> kick -> loose_ball_get -> handball -> receive,kick,True
7,8,free_kick -> handball -> receive -> kick -> loose_ball_get -> handball -> receive -> kick,end_possession,False


## 4. Check coverage and censoring

Only 2023â€“2025 are used during development. Reviewed episodes are excluded except when their only issue is an unknown ending at the end of the available recording. Internal transitions in those censored episodes remain usable, but their last prefixes have no target.

The coverage table counts source possessions after preprocessing. Entire quarters already quarantined for reversed timestamps are recorded in the source provenance, not silently reintroduced here. Excluding reviewed episodes is retrospective sample selection and may make this benchmark easier than unrestricted live data.

In [4]:
display(pd.read_csv(report_dir / "development_counts.csv"))
coverage = pd.read_csv(report_dir / "development_coverage.csv")
coverage_summary = coverage.groupby(["season", "included"]).possessions.sum().unstack(fill_value=0)
coverage_summary = coverage_summary.rename(columns={False: "excluded", True: "included"})
coverage_summary["excluded_percent"] = 100 * coverage_summary["excluded"] / coverage_summary.sum(axis=1)
display(coverage_summary.round(2))
display(pd.DataFrame([{
    key: audit[key] for key in [
        "source_events", "retained_events", "labelled_prefixes",
        "location_targets", "end_targets", "censored_last_prefixes"
    ]
}]))

,season,matches,possessions,observations,labelled_prefixes,locations
0,2023,216,53074,299425,298813,246351
1,2024,215,51282,291061,290459,239779
2,2025,216,51385,295159,294564,243774


included,excluded,included,excluded_percent
season,,,
2023,3086,53074,5.50
2024,2767,51282,5.12
2025,2650,51385,4.90


,source_events,retained_events,labelled_prefixes,location_targets,end_targets,censored_last_prefixes
0,935970,885645,883836,729904,153932,1809


## 5. Inspect the numerical inputs and architecture

The model embeds the action category and linearly projects ten numerical features. They describe the observed location, timing, quarter, displacement and normalised geometry relative to the attacking goal. The projection is linear, with no preceding ReLU.

The values below are the first three observed actions in an example from 2026. They are shown before standardisation. `dx_norm` and `dy_norm` use only the preceding observation. Geometry is measured in venue-normalised coordinates, not metres. Final scores, future actions, shot flags, endpoint reasons and total possession length are absent from the input.

Action embedding and numerical projection are concatenated, positional encoding is added, and a post-norm Transformer encoder processes the observed history. The last valid position feeds a linearâ€“ReLUâ€“linear head that outputs action logits and one coordinate pair. Right-padding is masked. Because every supplied token is already observed, the encoder can attend to the entire supplied prefix.

In [5]:
prefix = pd.read_csv(report_dir / "forecast_prefix.csv")
numeric = audit["numeric_features"]
display(prefix[["event_index", "action_type", *numeric]].round(3))
architecture = [
    ("Action embedding", config["embedding_dim"]),
    ("Numerical projection", config["continuous_dim"]),
    ("Combined encoder width", config["embedding_dim"] + config["continuous_dim"]),
    ("Attention heads", config["heads"]),
    ("Encoder layers", config["layers"]),
    ("Feed-forward width", (config["embedding_dim"] + config["continuous_dim"]) * config["ff_multiplier"]),
    ("Dense head width", config["dense_dim"]),
    ("Dropout", config["dropout"]),
    ("Action classes", len(audit["actions"])),
    ("Coordinate outputs", 2),
]
display(pd.DataFrame(architecture, columns=["component", "value"]))

,event_index,action_type,x_norm,y_norm,gap_seconds,elapsed_seconds,period_seconds,period,dx_norm,dy_norm,distance_to_goal_norm,angle_to_goal
0,0,loose_ball_get,-0.439,0.735,0,0,41,2,0.000,0.0,1.616,0.472
1,1,handball,-0.400,0.735,0,0,41,2,0.039,0.0,1.581,0.484
2,2,receive,-0.284,0.735,2,2,43,2,0.116,0.0,1.480,0.520


,component,value
0,Action embedding,16.0
1,Numerical projection,48.0
2,Combined encoder width,64.0
3,Attention heads,4.0
4,Encoder layers,2.0
5,Feed-forward width,256.0
6,Dense head width,64.0
7,Dropout,0.1
8,Action classes,13.0
9,Coordinate outputs,2.0


## 6. Review validation selection and refitting

Candidates fit on **2023â€“2024**. Six configurations are screened for up to four epochs, and two continue for up to twelve epochs with validation early stopping. The selection criterion is **5 Ã— multiclass cross-entropy + location RMSE**, computed on **2025**. Coordinate loss ignores terminal examples; action loss does not.

The selected architecture, history cap and epoch count are frozen. A new scaler and model are then fitted on **2023â€“2025**. Only after exporting and checking that model are 2026 inputs loaded. This is a bounded first search with one seed. It does not establish that this is the optimal architecture.

In [6]:
search = pd.read_csv(report_dir / "initial_search.csv")
config_table = pd.DataFrame(summary["plan"]["configs"]).rename_axis("candidate").reset_index()
display(config_table.merge(search[["candidate", "best_epoch", "selection_loss"]], on="candidate").round(5))
display(pd.read_csv(report_dir / "finalists.csv").round(5))
print("Selected candidate:", chosen_index)
print("Fresh refit epochs:", summary["selected"]["best_epoch"])
history = json.loads((report_dir / f"candidate-{chosen_index}-history.json").read_text(encoding="utf-8"))
display(pd.DataFrame([{
    "epoch": row["epoch"],
    "training_batch_objective": row["training_batch_objective"],
    "validation_log_loss": row["validation"]["action_log_loss"],
    "validation_location_RMSE": row["validation"]["location_rmse_norm"],
    "validation_selection_loss": row["validation"]["selection_loss"],
} for row in history]).round(5))
if summary["selected"]["best_epoch"] == summary["plan"]["final_epochs"]:
    print("The best epoch reached the search budget. Longer training has not been ruled out.")

,candidate,heads,ff_multiplier,dense_dim,learning_rate,weight_decay,max_history,embedding_dim,continuous_dim,layers,dropout,best_epoch,selection_loss
0,0,4,4,64,0.001,0.0001,16,8,24,1,0.10,4,4.14190
1,1,4,4,64,0.001,0.0001,32,8,24,1,0.10,4,4.14511
2,2,4,4,64,0.001,0.0001,64,8,24,1,0.10,4,4.14239
3,3,4,4,64,0.001,0.0001,32,16,48,1,0.10,4,4.11537
4,4,4,4,64,0.001,0.0001,32,16,48,2,0.10,4,4.09297
5,5,4,4,64,0.001,0.0001,32,16,48,2,0.25,4,4.10891


,candidate,best_epoch,examples,location_examples,action_log_loss,action_accuracy,top3_accuracy,macro_f1,multiclass_brier,location_rmse_norm,location_mean_distance_norm,selection_loss
0,4,8,294564,243774,0.78908,0.70569,0.95907,0.28834,0.41070,0.11404,0.11199,4.05942
1,5,8,294564,243774,0.79262,0.70503,0.95880,0.28584,0.41243,0.11438,0.11167,4.07750


Selected candidate: 4
Fresh refit epochs: 8


,epoch,training_batch_objective,validation_log_loss,validation_location_RMSE,validation_selection_loss
0,1,4.77052,0.82877,0.14143,4.28528
1,2,4.20712,0.80750,0.13909,4.17658
2,3,4.16131,0.79958,0.12111,4.11903
3,4,4.13069,0.79487,0.11865,4.09297
4,5,4.11174,0.79748,0.11697,4.10437
5,6,4.09836,0.79046,0.11336,4.06566
6,7,4.08555,0.79074,0.11669,4.07039
7,8,4.07563,0.78908,0.11404,4.05942
8,9,4.06934,0.79119,0.12318,4.07911
9,10,4.06301,0.79099,0.11187,4.06681


## 7. Compare action and location predictions

**Frequency + persistence** uses training action frequencies and predicts no location change. **Action transition** conditions next-action frequencies and mean displacement on the current action. Both are fitted on the same permitted data as the corresponding Transformer.

Lower action log loss and location RMSE are better. Higher accuracy and macro F1 are better. Macro F1 weights all thirteen declared action classes equally, including classes with zero target support; accuracy weights prefixes equally. Top-three accuracy checks whether the observed action appears among the three highest probabilities.

Location error is evaluated on all true continuation examples, including those whose action was misclassified. It is reported in normalised coordinate units. Each prefix contributes equal weight, unlike the earlier possession-weighted shot/goal experiments.

2026 is an already-observed project benchmark. These numbers must not be presented as evaluation on an untouched test set.

In [7]:
results = pd.read_csv(report_dir / "metrics.csv")
columns = ["split", "model", "examples", "action_log_loss", "action_accuracy",
           "top3_accuracy", "macro_f1", "multiclass_brier", "location_rmse_norm"]
display(results[columns].round(4))
test = results.loc[results.split.eq("test")].set_index("model")
reference = test.loc["transition"]
model_result = test.loc["transformer"]
print(f"Transformer action accuracy: {model_result.action_accuracy:.1%}")
print(f"Transition-baseline action accuracy: {reference.action_accuracy:.1%}")
print(f"Action log-loss change: {model_result.action_log_loss - reference.action_log_loss:+.4f}")
print(f"Location RMSE change: {model_result.location_rmse_norm - reference.location_rmse_norm:+.4f}")

,split,model,examples,action_log_loss,action_accuracy,top3_accuracy,macro_f1,multiclass_brier,location_rmse_norm
0,validation,transformer,294564,0.7891,0.7057,0.9591,0.2883,0.4107,0.1140
1,validation,frequency_persistence,294564,1.8446,0.2821,0.6562,0.0338,0.8121,0.1630
2,validation,transition,294564,0.8859,0.6372,0.9534,0.2173,0.4691,0.1409
3,test,transformer,304870,0.7833,0.7078,0.9599,0.2963,0.4083,0.1114
4,test,frequency_persistence,304870,1.8468,0.2779,0.6513,0.0335,0.8130,0.1625
5,test,transition,304870,0.8869,0.6334,0.9549,0.2158,0.4709,0.1413


Transformer action accuracy: 70.8%
Transition-baseline action accuracy: 63.3%
Action log-loss change: -0.1036
Location RMSE change: -0.0299


## 8. Inspect the full confusion matrix and rare actions

Rows are actual actions; columns are predicted actions. The first table shows counts. The second shows precision, recall, F1 and support for every action. This reveals failures that overall accuracy can hide, particularly for rare actions such as bounces. The kick-in classes have zero target support in this benchmark because their retained observations open episodes. Their zero F1 contributions remain included in the fixed-vocabulary macro average.

In [8]:
confusion = pd.read_csv(report_dir / "test-transformer-confusion.csv", index_col="actual")
display(confusion)
per_action = pd.read_csv(report_dir / "per_action.csv")
display(per_action.loc[
    per_action.split.eq("test") & per_action.model.eq("transformer"),
    ["action", "precision", "recall", "f1", "support"]
].round(4))
assert confusion.to_numpy().sum() == int(model_result.examples)

,bounce,free_kick,gather,ground_kick,handball,hard_ball_get,kick,kick_in,kick_in_play_on,loose_ball_get,mark,receive,end_possession
actual,,,,,,,,,,,,,
bounce,146,5,0,0,996,0,2149,0,0,0,0,0,0
free_kick,1,225,0,1,393,0,811,0,0,0,452,533,1512
gather,0,0,0,0,14,0,8,0,0,0,2219,481,2882
ground_kick,0,0,0,4,34,0,7,0,0,0,126,338,753
handball,54,13,0,0,42922,0,19530,0,0,0,0,39,22
hard_ball_get,0,0,0,0,54,0,12,0,0,0,417,474,1625
kick,68,51,0,0,18969,0,65629,0,0,0,0,1,3
kick_in,0,0,0,0,0,0,0,0,0,0,0,0,0
kick_in_play_on,0,0,0,0,0,0,0,0,0,0,0,0,0


,action,precision,recall,f1,support
39,bounce,0.5368,0.0443,0.0818,3296
40,free_kick,0.7525,0.0573,0.1065,3928
41,gather,0.0000,0.0000,0.0000,5604
42,ground_kick,0.6667,0.0032,0.0063,1262
43,handball,0.6602,0.6859,0.6728,62580
44,hard_ball_get,0.0000,0.0000,0.0000,2582
45,kick,0.7378,0.7746,0.7558,84721
46,kick_in,0.0000,0.0000,0.0000,0
47,kick_in_play_on,0.0000,0.0000,0.0000,0
48,loose_ball_get,0.0000,0.0000,0.0000,7369


## 9. Inspect one forecast

This example uses the first eligible 2026 episode with at least four observations and predicts after its third action. It was selected by a fixed rule, not because its prediction was successful.

The probability table describes the next retained action or an episode ending. The coordinate pair is conditional on continuation. It is a single point, not a separate location for each action and not a spatial uncertainty estimate.

In [9]:
saved_example = json.loads((report_dir / "forecast_example.json").read_text(encoding="utf-8"))
forecast = saved_example["forecast"]
display(prefix[["event_index", "action_type", "x_norm", "y_norm"]].round(3))
display(pd.Series(forecast["probabilities"], name="probability")
        .sort_values(ascending=False).to_frame().round(4))
print("Predicted location if continuation:", {
    key: round(value, 3) for key, value in forecast["location_if_continuation"].items()
})
print("Recorded next action:", saved_example["actual_next_action"])
print("Recorded next location:", saved_example["actual_location"])

,event_index,action_type,x_norm,y_norm
0,0,loose_ball_get,-0.439,0.735
1,1,handball,-0.400,0.735
2,2,receive,-0.284,0.735


,probability
handball,0.7187
kick,0.2377
bounce,0.0244
end_possession,0.0126
free_kick,0.0049
loose_ball_get,0.0011
hard_ball_get,0.0003
ground_kick,0.0002
gather,0.0001
receive,0.0000


Predicted location if continuation: {'x_norm': -0.209, 'y_norm': 0.751}
Recorded next action: kick
Recorded next location: {'x_norm': -0.21935483870967742, 'y_norm': 0.7058823529411765}


## 10. Optionally run the saved model

When local model weights exist, load them and run the same observed prefix. The API receives no future labels. On a fresh clone, this cell explains how to generate the local weights and leaves the saved report available for inspection.

In [10]:
model_dir = project_root / "artifacts" / "next-event-v1" / "model"
if (model_dir / "model.json").is_file():
    from afl_opposition.next_event_model import configure, load_model, forecast_next
    configure("cpu")
    model, scaler, metadata = load_model(model_dir)
    if metadata["fingerprint"] != saved_example["model_fingerprint"]:
        raise ValueError("The local model differs from this exported report.")
    live_forecast = forecast_next(model, scaler, prefix)
    np.testing.assert_allclose(
        list(live_forecast["probabilities"].values()),
        list(forecast["probabilities"].values()), atol=1e-6, rtol=1e-5,
    )
    print("Saved-model forecast reproduced.")
    print("Observed events:", live_forecast["observed_events"])
    print("Events used:", live_forecast["used_events"])
else:
    print("Local model weights are optional for reading this report.")
    print("To train: python -m afl_opposition.next_event_experiment")

Saved-model forecast reproduced.
Observed events: 3
Events used: 3


## What this supports next

This experiment compares a model using observed context with simple frequency and action-transition baselines. An improvement would not isolate the value of sequence history: the Transformer also receives richer numerical inputs and has greater capacity. A matched current-state neural model would be needed for that ablation. All teams share the model, but team identity does not yet affect its parameters or input embeddings.

An opposition-analysis extension could compare a particular team's observed choices with the shared model's expectations, then test whether adding team context improves out-of-season predictions. Tactical recommendations would also require football interpretation and data beyond the retained ball-control stream. This version does not infer off-ball movement, defensive formations, causal effects of decisions or multi-step match simulations.

The existing shot/goal models remain useful for a separate question: the likelihood of a possession producing a scoring opportunity or goal.